In [ ]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph , END, START
from langchain_core.messages import SystemMessage , HumanMessage
from typing import TypedDict ,Annotated,List
import operator

In [2]:

llm = ChatGroq(model="openai/gpt-oss-120b",api_key="X")

In [3]:
class DeepDriveState(TypedDict):
    raw_topic:str
    reserch_blueprint:str

    summary_section:str
    pros_cons_section:str
    fact_check_section:str
    case_study_section:str

    key_takeways:Annotated[List[str],operator.add]
    excution_logs:Annotated[List[str],operator.add]

    final_report:str

In [ ]:
def Topic_planner_node(state:DeepDriveState)->dict:
    """ Refine the raw topic into a strucural reserch blueprint"""
    topic=state['raw_topic']
    messge=[
        SystemMessage(
            content="You are a Senior Research Director. create a brief 3-point reserch scope for this topic",
        ),
        HumanMessage(content=f"topic->  {topic}")

    ]
    response=llm.invoke(messge)
    return{
        "reserch_blueprint":response.content,
        "Execution_logs":["[planner]-> Reserch blueprint created"]
    }


In [ ]:
def summarise_node(state:DeepDriveState)->dict:
    """ genrates a crisp summary and core working mechnism"""
    topic=state['raw_topic']
    blurprint=state['reserch_blueprint']
    message=[
        SystemMessage(
          content=("You are Technical Writer. write a crisp summary and explain how the topic works in simple bullts points"
                   "End your repsonse with a single line starting with Takeaway: summarizing the core essense "
          )
        ),
        HumanMessage(
            content=f"Topic:{topic}\n Blueprint:{blurprint}"
        )

    ]
    # Extract the takeway 
    takeway="Core concept Summarized"
    response=llm.invoke(message).content

    for line in response.split("\n"):
        if "Takeaway:" in line:
            takeway=line.replace("Takeaway:","").strip()
            break

    return{
        "summary_section":response,
        "Key_takeways":[f'summary Insight: {takeway}'],
        "execution_logs":["[paralle-1] Summarize node complete"]
    }

In [ ]:
def pros_cons_node(state:DeepDriveState)->dict:
    """"Extracts strenghts , weeknesses, and trade-offs"""
    topic=state['pros_cons_section']
    blueprint=state['reserch_blueprint']

    message=[
        SystemMessage(
            content=("You are the System analys , provide 4 pros and cons in clean markdwon table or bullet list  "
                     "End your response with a single ")
            
        )
    ]